# RBI site scraping — Praman data sourcing

Consolidates every scraping technique discovered while sourcing real RBI documents (see `../data-sources.md` for the full inventory and status). Two hard-won facts drive everything below:

1. **`rbi.org.in/scripts/*.aspx` pages are NOT gated.** Fetched directly with `requests` + a browser User-Agent, they return real content — no CAPTCHA, no bot-challenge, as long as the specific page doesn't depend on client-side JS (search, month/year dropdown navigation do; direct-ID lookups and default views don't).
2. **`rbidocs.rbi.org.in` (the actual PDF/xlsx file host) is uniformly bot-gated**, regardless of subpath (`/rdocs/notification/`, `/rdocs/Forms/`, `/rdocs/content/docs/` all tested). Every automated request gets HTTP 200 with a TSPD bot-challenge page disguised with the right file extension, not the real file. **The only way past it found so far is a real browser** (a human clicking the link, or a full browser-automation session) — plain `curl`/`requests`/`WebFetch` cannot get through it.

Keep these two facts in mind for every new document: try the `.aspx` page first (this notebook), fall back to a real browser only for `rbidocs.rbi.org.in` links.

In [1]:
from __future__ import annotations  # lets `str | None` etc. work on Python 3.9

import re
import subprocess
from pathlib import Path

import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
}

RAW_DIR = Path("../data/raw")
GATED_HOST = "rbidocs.rbi.org.in"  # confirmed bot-gated regardless of subpath — never fetch this with requests/curl

## Core helpers

In [2]:
def fetch(url: str, timeout: int = 30) -> str:
    """GET a rbi.org.in page. Raises if it looks like the gated host."""
    if GATED_HOST in url:
        raise ValueError(
            f"{GATED_HOST} is bot-gated for automated requests — confirmed dead end. "
            "Download this one manually in a real browser instead, then verify with is_real_pdf()."
        )
    resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    return resp.text


def html_to_text(html: str, start_marker: str | None = None) -> str:
    """Crude but effective HTML->text: strip script/style/tags, collapse whitespace,
    unescape the handful of entities RBI's pages actually use. Optionally trim
    everything before start_marker (e.g. skip the nav menu boilerplate)."""
    text = re.sub(r"<script.*?</script>", "", html, flags=re.S)
    text = re.sub(r"<style.*?</style>", "", text, flags=re.S)
    text = re.sub(r"<[^>]+>", " ", text)
    for entity, replacement in {
        "&rsquo;": "'", "&lsquo;": "'", "&rdquo;": '"', "&ldquo;": '"',
        "&ndash;": "-", "&mdash;": "-", "&amp;": "&", "&nbsp;": " ",
    }.items():
        text = text.replace(entity, replacement)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    if start_marker:
        idx = text.find(start_marker)
        if idx >= 0:
            text = text[idx:]
    return text.strip()


def is_real_pdf(path: Path) -> bool:
    """rbidocs.rbi.org.in serves the bot-challenge page with a .pdf extension and
    HTTP 200 — this is how we've been telling the difference: `file` reports the
    challenge page as HTML text, a real PDF as 'PDF document'."""
    result = subprocess.run(["file", str(path)], capture_output=True, text=True)
    return "PDF document" in result.stdout

## Specific page scrapers

Each mirrors a page we've already used successfully. `id=` values for Master Directions and notifications are stable per-document slots (not chronological/batch-assigned — confirmed the hard way scanning IDs 405–420 looking for a pattern that didn't exist).

In [3]:
def fetch_master_direction(doc_id: int) -> str:
    """e.g. fetch_master_direction(13641) -> MD 412, Fraud Risk Management."""
    html = fetch(f"https://www.rbi.org.in/Scripts/BS_ViewMasDirections.aspx?id={doc_id}")
    return html_to_text(html)


def fetch_notification(doc_id: int) -> str:
    """e.g. fetch_notification(13663) -> DoS.CO.PPG.66, the repeal circular."""
    html = fetch(f"https://www.rbi.org.in/scripts/NotificationUser.aspx?Id={doc_id}&Mode=0")
    return html_to_text(html, start_marker="Home Notifications")


def fetch_data_definitions() -> str:
    """The harmonized field-definition dictionary — India's DPM/MDRM equivalent,
    embedded directly in the page (NOT behind the gated .xlsx download)."""
    html = fetch("https://www.rbi.org.in/scripts/DataDefinition.aspx")
    return html_to_text(html, start_marker="DATA DEFINITIONS (Last Updated")


def fetch_returns_list() -> str:
    """'List of Returns Submitted to RBI' — 141+ entries, real download links
    (though most resolve to the gated host)."""
    html = fetch("https://www.rbi.org.in/scripts/BS_Listofallreturns.aspx")
    return html_to_text(html)


def fetch_withdrawn_circulars() -> str:
    """Full historical 'Circulars Withdrawn' table, all departments, back to 1999.
    No pagination, no JS needed for the table itself — confirmed by direct fetch.
    Useful as a search corpus: grep the result for a specific circular number."""
    html = fetch("https://www.rbi.org.in/Scripts/NotificationUserWithdrawnCircular.aspx")
    return html_to_text(html)


def fetch_circular_index_current_month() -> str:
    """BS_CircularIndexDisplay.aspx with NO query params defaults to the current
    month — that default view is real content. Month/year NAVIGATION on this page
    is onclick-JS-only though (confirmed) — can't reach other months this way."""
    html = fetch("https://www.rbi.org.in/Scripts/BS_CircularIndexDisplay.aspx")
    return html_to_text(html)

## Confirmed dead ends — don't retry these paths

- **RBI site search** (`SearchResults.aspx`): AJAX/postback-driven. Plain GET returns an empty form or a WAF 418 block on guessed param names (`k=`, `Title=`). Not solvable via static fetch.
- **Circular index month/year navigation**: genuinely `onclick`-only JS, not a URL parameter. Only the default (current month) view is reachable statically.
- **Any `rbidocs.rbi.org.in` link**: bot-gated uniformly. Real browser only — see `is_real_pdf()` above for verifying a manually-downloaded file actually came through.

## Re-run: refresh everything already sourced

Idempotent — re-fetches the pages behind every document currently in `data/raw/circulars/`. Useful if RBI updates a page (e.g. the withdrawn-circulars index eventually catching up with the July 2026 batch — rerun `fetch_withdrawn_circulars()` and search for `DoS.CO.PPG.66` again later).

In [4]:
SOURCED = {
    RAW_DIR / "circulars" / "RBI_DoS_2026-27_412_Commercial_Banks_Fraud_Risk_Management_Directions_2026.txt": lambda: fetch_master_direction(13641),
    RAW_DIR / "circulars" / "RBI_DoS_2026-27_415_Commercial_Banks_Supervisory_Returns_Directions_2026.txt": lambda: fetch_master_direction(13638),
    RAW_DIR / "circulars" / "RBI_Harmonised_Data_Definitions_2026-06-17.txt": fetch_data_definitions,
}

def refresh_all(dry_run: bool = True) -> None:
    for path, fetcher in SOURCED.items():
        text = fetcher()
        print(f"{path.name}: {len(text)} chars" + (" (dry run, not written)" if dry_run else ""))
        if not dry_run:
            path.write_text(text, encoding="utf-8")

# refresh_all(dry_run=False)  # uncomment to actually overwrite

## Search helper

How we actually confirmed the `DoS.CO.PPG.66` dead end — fetch the full page once, grep the plain text rather than guessing at query parameters.

In [5]:
def search_in(text: str, *terms: str, context: int = 80) -> list[str]:
    """Return context windows around each match of each term (case-sensitive,
    matches RBI's own formatting). Empty list = confirmed not present."""
    hits = []
    for term in terms:
        for m in re.finditer(re.escape(term), text):
            start = max(0, m.start() - context)
            end = min(len(text), m.end() + context)
            hits.append(text[start:end])
    return hits

# Example — re-verify the DoS.CO.PPG.66 dead end:
# withdrawn = fetch_withdrawn_circulars()
# search_in(withdrawn, "DoS.CO.PPG.66", "July 31, 2026", "628 circular")  # -> [] confirms still absent